In [ ]:
import pandas as pd
import re
from collections import defaultdict
import yaml

In [ ]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Data source: Incopat database

In [ ]:
npl_file = dataset_config['path_other'] + 'npl_citations_2023.csv'

In [ ]:
with open(npl_file) as fp:
    raw_csv = fp.readlines()

In [ ]:
raw_csv

In [ ]:
len(raw_csv)

In [ ]:
def contains_kw(text):
    """Return True if any digit is found in the text."""
    return bool(re.search(r'\d', text)) or bool(re.search(r'《|&lt;|》|&gt;', text)) or '全文' in text

def process_records(raw_csv):
    # Group lines by record ID (assuming contiguous records)
    grouped = defaultdict(list)
    for line in raw_csv:
        line = line.strip()
        if not line:
            continue
        try:
            rec_id, content = line.split("|", 1)
        except ValueError:
            continue
        grouped[rec_id].append(content.strip())
    
    # Merge contiguous lines until the merged string contains a digit
    result_records = []
    for rec_id, lines in grouped.items():
        line_iter = iter(lines)
        for line in line_iter:
            merged = line
            while not contains_kw(merged):
                try:
                    next_line = next(line_iter)
                except StopIteration:
                    break
                merged = merged.rstrip() + ", " + next_line.lstrip()
            result_records.append(f"{rec_id}|{merged}")
    
    # Remove all types of quotes from the records
    remove_chars = "\"'“”‘’"
    translation_table = str.maketrans('', '', remove_chars)
    return [record.translate(translation_table) for record in result_records]

In [ ]:
processed_csv = process_records(raw_csv)

In [ ]:
processed_csv

In [ ]:
def inspect_original(apn):
    for line in processed_csv:
        if line.startswith(apn):
            print(line)

In [ ]:
apn = 'CN201210237442.9'
inspect_original(apn)

In [ ]:
len(processed_csv)

In [ ]:
def npl_decoder(processed_csv):
    
    result = {'apn': [], 'record_to_recognize': []}
    result_special = []

    for s_row in processed_csv[1:]:
        processed_data = s_row.split('|')
        if len(processed_data) > 2:
            result_special.append(s_row)
        else:
            result['apn'].append(processed_data[0])
            result['record_to_recognize'].append(processed_data[1])

    return pd.DataFrame.from_dict(result), result_special

In [ ]:
npl_loaded, npl_special = npl_decoder(processed_csv)

In [ ]:
len(npl_special)

In [ ]:
npl_loaded

In [ ]:
def useful_record(record):
    import re
    rec = record.strip()
    # If too short, consider it not useful.
    if len(rec) < 20:
        return None
    
    # Define exclusion patterns (regex strings)
    exclusion_patterns = [
        r'^\s*(全文|同上)(?=.{0,5}$)$',           # exactly the full-text or same-as-above marker
        r'^(?=.{0,5})说明书.*页.*$',
        r'^\w{2}(.{0,20})说明书.*$', 
        r'^\w{2}(.{0,20})权利要求.*$', r'^\w{2}(.{0,20})摘要.*$', r'^\w{2}(.{0,20})实施例.*$', r'^\w{2}(.{0,20})全文.*$', 
        r'^\w{2}(.{0,5})\d{2,15}(.{0,5})第\d+(-\d+)?[页|栏].*$',
        r'^第\d+(-\d+)?页(.{0,25})',
        r'^(?=.{0,5})附图.*$',                  # only figure references
        r'^(?=.{0,5})表\d+.*$',                # only table references, such as table-number markers
        r'^\s*\d+\s*$',
        r'3GPP'
    ]
    for pat in exclusion_patterns:
        if re.search(pat, rec, re.IGNORECASE):
            return None

    # If none of the tests flagged the record, consider it useful.
    return record

use_index = npl_loaded.record_to_recognize.parallel_apply(useful_record).dropna().index
use_index

In [ ]:
npl_data = npl_loaded.loc[use_index]
npl_data

## Process rows in format 1 (with <>)

In [ ]:
def process_row_format1(record):
    import re
    from datetime import datetime

    pattern8 = re.compile(r'[.,;]\s*(\d{8})\s*[.,;]?')
    pattern4 = re.compile(r'[.,;]\s*(\d{4})\s*[.,;]?')

    def extract_year(s):
        match8 = pattern8.search(s)
        if match8:
            date_str = match8.group(1)
            try:
                dt = datetime.strptime(date_str, "%Y%m%d")
                if 1900 <= dt.year <= 2050:
                    return str(dt.year), match8.start()
            except ValueError:
                pass  # The date string is not a valid date.
    
        # Check for a 4-digit year.
        match4 = pattern4.search(s)
        if match4:
            year_val = int(match4.group(1))
            if 1900 <= year_val <= 2050:
                return str(year_val), match4.start()
        
        return None, None

    def detect_page_numbers(text: str) -> tuple[str, str]:
        # Chinese page style: must use the ordinal prefix and page suffix.
        pattern_chinese = r'第\s*(\d+)(?:\s*[-–]\s*(\d+))?\s*页'
        m = re.search(pattern_chinese, text)
        if m:
            first = m.group(1)
            last = m.group(2) if m.group(2) else first
            return first, last

        # Non-Chinese style: follows the pattern [.,;]\s*(\d+)\s*[.,;]?
        # Extended to capture an optional range with a dash.
        pattern_non_chinese = r'[.,;:，：]\s*(\d+)\s*[-–]\s*(\d+)\s*[.,;]?'
        m = re.search(pattern_non_chinese, text)
        if m:
            first = m.group(1)
            last = m.group(2) if m.group(2) else first
            return first, last

        return None, None
    
    def remove_html_symbols(text):
        pattern = r'&[A-Za-z0-9#]+;|<[^>]+>'
        return re.sub(pattern, '', text)
    
    def extract_clean_parts(text: str):
        # Pattern to match a marker with optional punctuation (dot, comma, semicolon, space)
        # before and after the marker.
        inner_pattern = r'(?:[ .,;]+)?(?:《|&lt;)(.*?)(?:》|&gt;)(?:[ .,;]+)?'
        
        # Find all inner matches.
        inner_matches = re.findall(inner_pattern, text)
        
        # Use a similar pattern (without capturing the inner part) to split the text.
        split_pattern = r'(?:[ .,;]+)?(?:《|&lt;).*?(?:》|&gt;)(?:[ .,;]+)?'
        parts = re.split(split_pattern, text)
        
        # The first part is the prefix and the last part is the suffix.
        prefix = parts[0].rstrip(' .,;') if parts else ""
        suffix = parts[-1].lstrip(' .,;') if parts else ""
        
        return remove_html_symbols(prefix), inner_matches, remove_html_symbols(suffix)
    

    # Initialize result dictionary with keys set to None
    result = {
        'paper_title': None,
        'authors': None,
        'journal_name': None,
        'year': None,
        'first_page': None,
        'last_page': None
    }

    if '&lt;' in record or '&gt;' in record or '《' in record or '》' in record:

        record_L, inners, record_R = extract_clean_parts(record)

        if len(inners) > 2:
            return result
        elif len(inners) == 2:
            result["paper_title"] = inners[0]
            result["journal_name"] = inners[1]
            result["authors"] = record_L
        elif len(inners) == 1:
            result["journal_name"] = inners[0]

            if ';' in record_L:
                str_split = ';'
            else:
                str_split = '.'
        
            parts_L = [p.strip() for p in record_L.split(str_split) if p.strip()]
            if len(parts_L) >= 2:
                result["authors"] = ', '.join(parts_L[1:]) if str_split == ';' else '. '.join(parts_L[0:-1])
                result["paper_title"] = parts_L[0] if str_split == ';' else parts_L[-1]
            if len(parts_L) == 1:
                result["authors"] = record_L
                result["paper_title"] = inners[0]
                result["journal_name"] = None

        year, year_start = extract_year(';' + record_R)
        result['year'] = year
        
        if len(inners) == 1 and len(parts_L) == 1 and year:
            record_Rx = record_R[:year_start]
            if '.' in record_Rx:
                result["journal_name"] = record_Rx.split('.')[0]

        first_page, last_page = detect_page_numbers(';' + record_R)
        if first_page and last_page and int(first_page) <= int(last_page):
            result['first_page'] = first_page
            result['last_page'] = last_page

    return result


In [ ]:
process_row_format1('CN202010860531.3|A Deep Reinforcement Learning Based Approach for Energy-Efficient Channel Allocation in Satellite Internet of Things; Baokang Zhao，etc.; 《IEEE Access》; 20200326; 20-21')

In [ ]:
process_row_format1('章丽华.口服泡腾片剂及其处方设计.《国外医药-合成药、生化药、制剂分册》.1991, 第12卷(第2期),11-12')

In [ ]:
decoded_format1 = npl_data.record_to_recognize.parallel_apply(process_row_format1)
decoded_format1 = pd.DataFrame.from_dict(decoded_format1.to_dict(), orient='index')

In [ ]:
npl_decoded_format1 = decoded_format1.dropna(subset=['paper_title', 'authors'])
npl_decoded_format1['sources'] = 'format1'
npl_decoded_format1

In [ ]:
decoded_format1.loc[list(set(decoded_format1.dropna(subset=['paper_title', 'authors']).index) - set(decoded_format1.dropna(subset=['paper_title', 'authors', 'journal_name']).index))].drop_duplicates()

## Process rows in format 2 (Without <>, separated by dot)

In [ ]:
not_recognized = npl_data[~npl_data.index.isin(npl_decoded_format1.index)]
len(not_recognized)

In [ ]:
def check_http(text):
    import re
    url_pattern = r'https?\s*:\s*/\s*/(?:\s*\S+)+[.,;]?'
    if re.search(url_pattern, text):
        return text

drop_index = not_recognized.record_to_recognize.parallel_apply(check_http).dropna().index
drop_index

In [ ]:
to_search_format2 = not_recognized[~not_recognized.index.isin(drop_index)]
len(to_search_format2)

In [ ]:
to_search_format2.iloc[:50].record_to_recognize.to_list()

In [ ]:
to_search_format2[to_search_format2.record_to_recognize.parallel_apply(lambda x: ';' in x)].record_to_recognize.to_list()

In [ ]:
def process_row_format2(record):
    import re
    from datetime import datetime

    pattern_year = re.compile(r'[.,;]\s*(\d{4})\s*[.,;]')

    def extract_year(s):
        match = pattern_year.search(s)
        if match:
            year_val = int(match.group(1))
            if 1900 <= year_val <= 2050:
                return str(year_val), match.start(), match.end()
        
        return None, None, None

    def detect_page_numbers(text: str) -> tuple[str, str]:
        # Chinese page style: must use the ordinal prefix and page suffix.
        pattern_chinese = r'第\s*(\d+)(?:\s*[-–]\s*(\d+))?\s*页'
        m = re.search(pattern_chinese, text)
        if m:
            first = m.group(1)
            last = m.group(2) if m.group(2) else first
            return first, last

        # Non-Chinese style: follows the pattern [.,;]\s*(\d+)\s*[.,;]?
        # Extended to capture an optional range with a dash.
        pattern_non_chinese = r'[.,;:，：]\s*(\d+)\s*[-–]\s*(\d+)\s*[.,;]?'
        m = re.search(pattern_non_chinese, text)
        if m:
            first = m.group(1)
            last = m.group(2) if m.group(2) else first
            return first, last

        return None, None
    
    def remove_html_symbols(text):
        pattern = r'&[A-Za-z0-9#]+;|<[^>]+>'
        return re.sub(pattern, '', text)

    def split_entry(entry):
        entry = entry.strip()
        
        # Normalize punctuation around special tokens (et al. and the Chinese and-others marker).
        # This version allows an optional preceding comma and ensures spaces around the token,
        # removing extra dots that follow the token.
        entry = re.sub(r'\s*,?\s*(et al\.)\.+', r' \1 ', entry, flags=re.IGNORECASE)
        entry = re.sub(r'\s*,?\s*(等)\.+', r' \1 ', entry)
        entry = entry.strip()
        
        def valid_separator(match, text):
            pos = match.start()
            # Avoid splitting on initials like "C." by checking the preceding characters.
            if re.search(r'\b[A-Za-z]\.$', text[max(0, pos-3): pos+1]):
                return False
            following = text[match.end():]
            # Only allow a separator if the following text starts with a letter (or a Chinese character).
            if re.search(r'^\s*([A-Za-z\u4e00-\u9fff])', following):
                return True
            return False

        def split_by_delim(text):
            # First, try to find a double-dot delimiter (“..”) that is not part of three or more dots.
            m = re.search(r'\.\.(?!\.)', text)
            if m:
                return text[:m.start()].strip(), text[m.end():].strip()
            # Otherwise, iterate over single dots and choose the first valid separator.
            for m in re.finditer(r'\.', text):
                if valid_separator(m, text):
                    return text[:m.start()].strip(), text[m.end():].strip()
            return text, ''
        
        # If a special token (et al. or the Chinese and-others marker) exists, force authors to end right after that token.
        special = re.search(r'(et al\.|等)', entry, re.IGNORECASE)
        if special:
            authors = entry[:special.end()].strip()
            remainder = entry[special.end():].strip()
            # Remove any leading punctuation (dots or commas) from the remainder.
            remainder = re.sub(r'^[\.,]+', '', remainder).strip()
        else:
            authors, remainder = split_by_delim(entry)
        
        # Split the remainder into title and journal.
        title, journal = split_by_delim(remainder)
        # Clean up trailing punctuation, spaces, or digits from the journal.
        journal = re.sub(r'[\s\d\W]+$', '', journal)
        
        return authors, title, journal


    result = {
        'paper_title': None,
        'authors': None,
        'journal_name': None,
        'year': None,
        'first_page': None,
        'last_page': None
    }

    record = remove_html_symbols(record).replace('etc.', '')
    year, yr_start, yr_end = extract_year(record)

    if year:
        record_L = record[:yr_start]
        record_R = record[yr_end:]
    else:
        yr_match = re.search(r'[.,;]\s*(\d{4})$', record)
        if yr_match and yr_match.end() == len(record):
            return result
        record_L = '.'.join(record.split('.')[:-1])
        record_R = record.split('.')[-1]

    first_page, last_page = detect_page_numbers(';' + record_R)
    if first_page and last_page and int(first_page) <= int(last_page):
        result['first_page'] = first_page
        result['last_page'] = last_page
    #return record_L, year, record_R, result['first_page'], result['last_page']

    if '..' in record_L:
        str_split = '..'
    else:
        str_split = '.'
    parts_L = [p.strip() for p in record_L.split(str_split) if p.strip()]
    if year and len(parts_L) in [2, 3]:
        result["authors"] = parts_L[0]
        result["paper_title"] = parts_L[1]
        if len(parts_L) == 3:
            result['journal_name'] = parts_L[2]
        return result
    
    authors, title, journal = split_entry(record_L)
    result["authors"] = authors.strip() if authors.strip() else None
    result["paper_title"] = title.strip() if title.strip() else None
    result['journal_name'] = journal.strip() if journal.strip() else None


    return result

In [ ]:
decoded_format2 = to_search_format2.record_to_recognize.parallel_apply(process_row_format2)
decoded_format2 = pd.DataFrame.from_dict(decoded_format2.to_dict(), orient='index')

In [ ]:
npl_decoded_format2 = decoded_format2.dropna(subset=['paper_title', 'authors'])
npl_decoded_format2['sources'] = 'format2'
npl_decoded_format2

In [ ]:
to_search_format2.loc[2004].record_to_recognize

In [ ]:
process_row_format2('水稻氮素营养水平与光谱特性的关系. 周启发，王人潮.浙江大学学报(农业与生命科学版)，第19卷第S1期. 1993')

## Process rows in format 3 (Chinese entries separated by semi-colon)

In [ ]:
not_recognized2 = to_search_format2[~to_search_format2.index.isin(npl_decoded_format2.index)]
len(not_recognized2)

In [ ]:
to_search_format3 = not_recognized2
len(to_search_format3)

In [ ]:
def process_row_format3(record):
    import re
    from datetime import datetime

    def extract_year(s):
        pattern8 = r'\s*(\d{8})\s*'
        pattern4 = r'\s*(\d{4})\s*'

        match8 = re.search(pattern8, s)
        if match8:
            date_str = match8.group(1)
            try:
                dt = datetime.strptime(date_str, "%Y%m%d")
                if 1900 <= dt.year <= 2050:
                    return str(dt.year), match8.start()
            except ValueError:
                pass  # The date string is not a valid date.
    
        # Check for a 4-digit year.
        match4 = re.search(pattern4, s)
        if match4:
            year_val = int(match4.group(1))
            if 1900 <= year_val <= 2050:
                return str(year_val), match4.start()
        
        return None, None

    def detect_page_numbers(text):
        # Chinese page style: must use the ordinal prefix and page suffix.
        pattern_chinese = r'第\s*(\d+)(?:\s*[-–]\s*(\d+))?\s*页'
        m = re.search(pattern_chinese, text)
        if m:
            first = m.group(1)
            last = m.group(2) if m.group(2) else first
            return first, last

        # Non-Chinese style: follows the pattern [.,;]\s*(\d+)\s*[.,;]?
        # Extended to capture an optional range with a dash.
        pattern_non_chinese = r'\s*(\d+)[-–](\d+)\s*[.,;+]?\d+$'
        m = re.search(pattern_non_chinese, text)
        if m:
            first = m.group(1)
            last = m.group(2) if m.group(2) else first
            return first, last

        return None, None
    
    def remove_html_symbols(text):
        pattern = r'&[A-Za-z0-9#]+;|<[^>]+>'
        return re.sub(pattern, '', text)

    result = {
        'paper_title': None,
        'authors': None,
        'journal_name': None,
        'year': None,
        'first_page': None,
        'last_page': None
    }

    record = remove_html_symbols(record)
    if ';' in record:
        parts = [p.strip() for p in record.strip().split(';') if p.strip()]
        if len(parts) < 3:
            return result

        first_page, last_page = detect_page_numbers(parts[-1])
        if first_page and last_page:
            if int(first_page) <= int(last_page):
                result["first_page"] = first_page
                result["last_page"] = last_page
            parts.pop()
        elif parts[-1] == '全文':
            parts.pop()
        else:
            result['paper_title'] = parts[0]
            result['authors'] = parts[1]
            return result

        if re.match(r'^(?:第)?\d+.*$', parts[-1]):
            parts.pop()

        year, _ = extract_year(parts[-1])
        if year:
            result["year"] = year
            parts.pop()

        if len(parts) >= 3:
            result["journal_name"] = parts[-1]
            result["authors"] = ', '.join(parts[1:-1])
            result["paper_title"] = parts[0]
                
        if not result["paper_title"] and len(parts) >= 2:
            result['paper_title'] = parts[0]
            result['authors'] = parts[1]
            return result

    return result

In [ ]:
process_row_format3('BCCWJ-DepPara :  A Syntactic Annotation Treebank on the Balanced Corpus of Contemporary Written Japanese; Masayuki Asahara et al.; The COLING 2016 Organizing Committee; 第49-58页')

In [ ]:
decoded_format3 = to_search_format3.record_to_recognize.parallel_apply(process_row_format3)
decoded_format3 = pd.DataFrame.from_dict(decoded_format3.to_dict(), orient='index')

In [ ]:
npl_decoded_format3 = decoded_format3.dropna(subset=['paper_title', 'authors',])
npl_decoded_format3['sources'] = 'format3'
npl_decoded_format3

In [ ]:
to_search_format3.loc[2365125].record_to_recognize

## Process rows in format 4 (Chinese entries separated by dot and space)

In [ ]:
not_recognized3 = to_search_format3[~to_search_format3.index.isin(npl_decoded_format3.index)]
len(not_recognized3)

In [ ]:
to_search_format4 = not_recognized3
len(to_search_format4)

In [ ]:
def process_row_format4(record):
    import re
    from datetime import datetime

    def ends_with_year(s):
        pattern = r'\.\s+(19\d{2}|20[0-4]\d|2050)$'
        return re.search(pattern, s)
    
    def contains_chinese(text: str) -> bool:
        return bool(re.search(r'[\u4e00-\u9fff]', text))
    
    def remove_html_symbols(text):
        pattern = r'&[A-Za-z0-9#]+;|<[^>]+>'
        return re.sub(pattern, '', text)
    
    def separate_reference(record):
        record = re.sub(r'\s*\.*\s*\d{4}\s*$', '', record.strip())
        segments = [seg.strip() for seg in record.split('.') if seg.strip()]
        
        if len(segments) < 3:
            return None, None, None
        
        title = segments[0]
        
        journal_start = None
        for i in range(1, len(segments)):
            if re.search(r'\b(vol|no)\b', segments[i], re.IGNORECASE):
                journal_start = i
                break
        
        if journal_start is None:
            journal_start = len(segments) - 1
        
        authors_segments = segments[1:journal_start]
        authors = ', '.join(authors_segments) if authors_segments else ''
        
        journal_segments = segments[journal_start:]
        journal = '. '.join(journal_segments)
        
        return title, authors, journal

    result = {
        'paper_title': None,
        'authors': None,
        'journal_name': None,
        'year': None,
        'first_page': None,
        'last_page': None
    }

    record = remove_html_symbols(record)

    year_match = ends_with_year(record)
    if year_match:
        result['year'] = year_match.group(1)
    else:
        return result

    title, authors, journal = separate_reference(record)
    
    if title and authors and journal:
        result['paper_title'] = title
        result['authors'] = authors
        result['journal_name'] = journal
    else:
        parts = record[:year_match.start()].split('.')
        if len(parts) == 1 or len(parts) == 0 or parts[0][-1:] == '等':
            return result
        result['paper_title'] = parts[0]
        author_1 = parts[1].split('，')
        author_2 = parts[1].split(',')
        if author_1 or author_2:
            result['authors'] = author_1[0] if author_1[0] < author_2[0] else author_2[0]

    return result

In [ ]:
process_row_format4('淀粉与淀粉制品工艺学.  高嘉安，5, 12, 19，中国农业出版社.  2001')

In [ ]:
process_row_format4('Colorimetric detection of specific DNA segments amplified bypolymerase chain reaction. Kemp DJ, et al.Proc Natl Acad Sci U S A, Vol.86 No.7. 1989')

In [ ]:
decoded_format4 = to_search_format4.record_to_recognize.parallel_apply(process_row_format4)
decoded_format4 = pd.DataFrame.from_dict(decoded_format4.to_dict(), orient='index')

In [ ]:
npl_decoded_format4 = decoded_format4.dropna(subset=['paper_title', 'authors'])
npl_decoded_format4['sources'] = 'format4'
npl_decoded_format4

In [ ]:
to_search_format4.loc[1902095].record_to_recognize

## Process rows in format 5 (Chinese entries separated by only space)

In [ ]:
not_recognized4 = to_search_format4[~to_search_format4.index.isin(npl_decoded_format4.index)]
len(not_recognized4)

In [ ]:
to_search_format5 = not_recognized4
len(to_search_format5)

In [ ]:
def process_row_format5(record):
    import re

    def ends_with_year(s):
        pattern = r'\s+(19\d{2}|20[0-4]\d|2050)$'
        return re.search(pattern, s)
    
    def remove_html_symbols(text):
        pattern = r'&[A-Za-z0-9#]+;|<[^>]+>'
        return re.sub(pattern, '', text)

    def split_reference(ref):
        # Regex pattern to find a space between two Chinese characters.
        pattern = r"[^，]\.?\s+(?=[\u4e00-\u9fff])"
        matches = list(re.finditer(pattern, ref))
        
        if not matches:
            return ref.strip(), "", ""
        
        # Use the last occurrence of the matching space.
        last_match = matches[-1]
        split_index = last_match.start() + 1
        title = ref[:split_index]
        remainder = ref[split_index + 1:].strip()
        
        # Try to split the remainder using the Chinese and-others marker followed by a comma.
        # The regex pattern here captures authors ending with the Chinese and-others marker and then a comma.
        match_authors = re.search(r"^(.*?等)，(.*)$", remainder)
        if match_authors:
            authors = match_authors.group(1)
            journal = match_authors.group(2)
        else:
            # Fall back: split at the first comma "，"
            match_comma = re.search(r"^(.*?)，(.*)$", remainder)
            if match_comma:
                authors = match_comma.group(1)
                journal = match_comma.group(2)
            else:
                authors = remainder
                journal = ""
        
        return title.strip(), authors.strip(), journal.strip()


    result = {
        'paper_title': None,
        'authors': None,
        'journal_name': None,
        'year': None,
        'first_page': None,
        'last_page': None
    }

    record = remove_html_symbols(record)

    year_match = ends_with_year(record)
    if year_match:
        result['year'] = year_match.group(1)
    else:
        return result
    
    record_x = record[:year_match.start()]
    match_dot = re.search(r'(?<=[\u4e00-\u9fff])\.(?=[\u4e00-\u9fff])', record_x)
    if match_dot:
        title_author = record_x[:match_dot.start()]
        if len(title_author.split()) <= 1:
            return result
        title = title_author.split()[0]
        authors = title_author.split()[1]
        journal = record_x[match_dot.end():]
    else:
        title, authors, journal = split_reference(record_x)
    
    if title and authors and journal:
        result['paper_title'] = title
        result['authors'] = authors
        result['journal_name'] = journal

    return result

In [ ]:
process_row_format5('计算机控制激光器系统研究 于贵明等，大连理工大学学报，第37卷第增刊2期 1997')

In [ ]:
process_row_format5('助磨剂对水泥性能的影响  陶珍东，郑少华等.硅酸盐通报，第5期  2002')

In [ ]:
decoded_format5 = to_search_format5.record_to_recognize.parallel_apply(process_row_format5)
decoded_format5 = pd.DataFrame.from_dict(decoded_format5.to_dict(), orient='index')

In [ ]:
npl_decoded_format5 = decoded_format5.dropna(subset=['paper_title', 'authors'])
npl_decoded_format5['sources'] = 'format5'
npl_decoded_format5

## Result output

### (1) Not recognized (using LLM in the next step)

In [ ]:
not_recognized5 = to_search_format5[~to_search_format5.index.isin(npl_decoded_format5.index)]
not_recognized5

In [ ]:
not_recognized5.to_parquet(dataset_config['path_processed'] + 'CN_CN/CN1_NPL_IncoPat_ND.parquet', index=None)

### (2) Recognized in this step

In [ ]:
final_result = pd.concat([npl_decoded_format1, npl_decoded_format2, npl_decoded_format3, npl_decoded_format4, npl_decoded_format5])
final_result['apn'] = npl_loaded.loc[final_result.index].apn
final_result = final_result[['apn', 'paper_title', 'authors', 'journal_name', 'year', 'first_page', 'last_page', 'sources']].drop_duplicates().sort_index()
final_result

In [ ]:
final_result['year'] = final_result['year'].astype(pd.Int16Dtype())
final_result['first_page'] = final_result['first_page'].astype(pd.Int16Dtype())
final_result['last_page'] = final_result['last_page'].astype(pd.Int16Dtype())

In [ ]:
final_result

In [ ]:
final_result.to_parquet(dataset_config['path_processed'] + 'CN_CN/CN1_NPL_IncoPat_Formatted.parquet')